This downloads all files from TEMPO locally
In this case I chose the california reference scenario modelling in 2024

In [1]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os

# Set up the S3 client with unsigned (public) access
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

bucket_name = 'nrel-pds-dsgrid'
# https://data.openei.org/s3_viewer?bucket=nrel-pds-dsgrid&prefix=tempo%2Ftempo-2022%2Fv1.0.0%2Ffull_dataset%2Ftable.parquet%2Fscenario%3Dreference%2Ftempo_project_model_years%3D2024%2Fstate%3DCA%2F
prefix = 'tempo/tempo-2022/v1.0.0/full_dataset/table.parquet/scenario=reference/tempo_project_model_years=2024/state=CA/'

# Local folder to save files
local_dir = './nrel_tempo_ca_2024'
os.makedirs(local_dir, exist_ok=True)

# List objects under the prefix
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

if 'Contents' not in response:
    print("No files found.")
else:
    for obj in response['Contents']:
        key = obj['Key']
        if key.endswith('.parquet'):
            filename = os.path.basename(key)
            local_path = os.path.join(local_dir, filename)
            print(f"Downloading {key} to {local_path}")
            s3.download_file(bucket_name, key, local_path)

Within the above parquet file, I am checking the unique values for all parameters

In [2]:
import pandas as pd

# Load one Parquet file
file_path = './nrel_tempo_ca_2024/part-00023-d668d769-0c35-46e0-94bc-3fa450b31efc.c000.snappy.parquet'  # Replace with your actual file name
df = pd.read_parquet(file_path)

# List of columns to check
columns_to_check = [
    'time_est',             # Timestamp
    'end_use',              # Should be 'EVCharging'
    'household_and_vehicle_type',
    'transportation',       # E.g., "LDV", "MDV"
    'weather_2012',         # Weather scenario (should be a string or int)
    'county'                # County name
]

# Count unique values per column
for col in columns_to_check:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")
    else:
        print(f"{col}: column not found in DataFrame")


time_est: 8784 unique values
end_use: 2 unique values
household_and_vehicle_type: 720 unique values
transportation: 1 unique values
weather_2012: 1 unique values
county: 4 unique values


Checking what each unique parameter is for household and vehicle type as well as the end uses within again this same parquet file

In [3]:
import pandas as pd

# Load one Parquet file
file_path = './nrel_tempo_ca_2024/part-00023-d668d769-0c35-46e0-94bc-3fa450b31efc.c000.snappy.parquet'  # Replace with your exact filename
df = pd.read_parquet(file_path)

# Show unique values (no repetitions)
household_vehicle_types = df['household_and_vehicle_type'].dropna().unique().tolist()
end_uses = df['end_use'].dropna().unique().tolist()

print("Household and Vehicle Types:")
for item in household_vehicle_types:
    print(f"- {item}")

print("\nEnd Uses:")
for item in end_uses:
    print(f"- {item}")


Household and Vehicle Types:
- Some_Drivers_Smaller+High_Income+Small_Town+SUV+BEV_100
- Some_Drivers_Larger+Middle_Income+Small_Town+SUV+PHEV_50
- Some_Drivers_Larger+Middle_Income+Rural+Pickup+BEV_100
- Some_Drivers_Smaller+Low_Income+Rural+SUV+BEV_100
- Some_Drivers_Smaller+High_Income+Small_Town+Midsize+PHEV_25
- Some_Drivers_Smaller+Middle_Income+Rural+Midsize+BEV_100
- Some_Drivers_Larger+Low_Income+Rural+Midsize+BEV_100
- Single_Driver+Middle_Income+Rural+SUV+BEV_100
- Single_Driver+Middle_Income+Small_Town+Compact+BEV_100
- Some_Drivers_Smaller+High_Income+Rural+SUV+PHEV_50
- Some_Drivers_Larger+High_Income+Rural+SUV+PHEV_50
- Single_Driver+Middle_Income+Small_Town+SUV+PHEV_50
- Some_Drivers_Larger+High_Income+Rural+Midsize+PHEV_25
- Single_Driver+Middle_Income+Rural+Midsize+PHEV_25
- Some_Drivers_Larger+Low_Income+Small_Town+SUV+BEV_300
- Some_Drivers_Larger+Low_Income+Small_Town+Compact+BEV_100
- Some_Drivers_Smaller+Low_Income+Small_Town+Midsize+BEV_100
- Some_Drivers_Larger

Make sure I have my 58 counties in these 20 parquet files.

In [4]:
import os
import pandas as pd

# Path to your folder with the 20 files
folder_path = './nrel_tempo_ca_2024'

# List all parquet files
parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

# Set to collect all unique counties
unique_counties = set()

# Loop through each file and extract unique counties
for file in parquet_files:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_parquet(file_path, columns=['county'])  # Only read the "county" column for efficiency
        unique_counties.update(df['county'].dropna().unique())
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Output the result
print(f"\n✅ Total unique counties: {len(unique_counties)}")
print("List of counties:")
for county in sorted(unique_counties):
    print(f"- {county}")



✅ Total unique counties: 58
List of counties:
- 06001
- 06003
- 06005
- 06007
- 06009
- 06011
- 06013
- 06015
- 06017
- 06019
- 06021
- 06023
- 06025
- 06027
- 06029
- 06031
- 06033
- 06035
- 06037
- 06039
- 06041
- 06043
- 06045
- 06047
- 06049
- 06051
- 06053
- 06055
- 06057
- 06059
- 06061
- 06063
- 06065
- 06067
- 06069
- 06071
- 06073
- 06075
- 06077
- 06079
- 06081
- 06083
- 06085
- 06087
- 06089
- 06091
- 06093
- 06095
- 06097
- 06099
- 06101
- 06103
- 06105
- 06107
- 06109
- 06111
- 06113
- 06115


Making sure each county is only present in one parquet file. 

In [5]:
import os
import pandas as pd
from collections import defaultdict

# Folder containing the 20 parquet files
folder_path = './nrel_tempo_ca_2024'

# List of all parquet files in the folder
parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

# Dictionary to map county -> list of filenames it's found in
county_to_files = defaultdict(list)

# Loop over all files
for file in parquet_files:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_parquet(file_path, columns=['county'])
        counties = df['county'].dropna().unique()
        for county in counties:
            county_to_files[county].append(file)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Check for duplicates
print("\n🔍 Checking for counties in multiple files:")
duplicate_counties = {county: files for county, files in county_to_files.items() if len(files) > 1}

if duplicate_counties:
    print(f"\n⚠️ The following counties appear in multiple files:")
    for county, files in duplicate_counties.items():
        print(f"- {county} appears in {len(files)} files: {files}")
else:
    print("\n✅ All counties appear in only one file.")



🔍 Checking for counties in multiple files:

✅ All counties appear in only one file.


Test run to see the load profile over a whole year and also the average daily load profile for all the data for a county specifically. This will be searching in all 20 parquet files, and create an excel file.

In [ ]:
import pandas as pd
import os
from glob import glob

# Path to all 20 files
folder_path = './nrel_tempo_ca_2024'
file_paths = glob(os.path.join(folder_path, '*.parquet'))

# Target county
target_county = '06033'

# Collect matching data from all files
dfs = []
for path in file_paths:
    df = pd.read_parquet(path, columns=['time_est', 'county', 'value'])
    df_filtered = df[df['county'] == target_county]
    if not df_filtered.empty:
        dfs.append(df_filtered)

# Combine all relevant data
df_all = pd.concat(dfs, ignore_index=True)

# YEARLY LOAD PROFILE
yearly_profile = df_all.groupby('time_est')['value'].sum().reset_index()

# AVERAGE DAILY LOAD PROFILE
df_all['hour'] = pd.to_datetime(df_all['time_est']).dt.hour
average_daily_profile = df_all.groupby('hour')['value'].mean().reset_index()

# Rename columns for clarity
yearly_profile.columns = ['timestamp', 'load_kWh']
average_daily_profile.columns = ['hour_of_day', 'avg_load_kWh']

# # Save both to Excel
# with pd.ExcelWriter('county_06033_load_profiles.xlsx') as writer:
#     yearly_profile.to_excel(writer, index=False, sheet_name='Yearly Profile')
#     average_daily_profile.to_excel(writer, index=False, sheet_name='Average Daily Profile')

# print("✅ Load profiles saved to 'county_06033_load_profiles.xlsx'")

✅ Load profiles saved to 'county_06033_load_profiles.xlsx'


Plot yealry and average daily loads for all car and house types for a single county

In [98]:
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt

# Path to all 20 files
folder_path = './nrel_tempo_ca_2024'
file_paths = glob(os.path.join(folder_path, '*.parquet'))

# Target county
target_county = '06001'
end_use = "electricity_ev_l1l2"
type = "Some_Drivers_Smaller+Middle_Income+Small_Town+SUV+BEV_300"

# Collect matching data from all files
dfs = []
for path in file_paths:
    df = pd.read_parquet(path)
    df_filtered = df[(df['county'] == target_county) &
                     (df['end_use'] == end_use) &
                     (df['household_and_vehicle_type'] == type)
                    ]
    if not df_filtered.empty:
        dfs.append(df_filtered)

# Combine all relevant data
df_all = pd.concat(dfs, ignore_index=True)

# print(df_all.head())

# Convert time_est to datetime once for all
df_all['time_est'] = pd.to_datetime(df_all['time_est'])

# YEARLY LOAD PROFILE (daily sums)
df_all['date'] = df_all['time_est'].dt.date
daily_profile = df_all.groupby('date')['value'].sum().reset_index()
daily_profile.columns = ['date', 'daily_load_kWh']

df_all["date"] = df_all["time_est"].dt.floor("H").dt.date
df_all["hour"] = df_all["time_est"].dt.floor("H").dt.hour

counts_by_date_hour = (
    df_all.groupby(["date", "hour"])
          .agg(
              n_entries=("value", "size"),
              hourly_total_value=("value", "sum")  # total load in kWh for that date+hour
          )
          .reset_index()
)

print(counts_by_date_hour.head(10))  # preview

# How many (date, hour) combos exist?
total_unique_date_hour = counts_by_date_hour.shape[0]
print("Total unique (date, hour) combinations:", total_unique_date_hour)

# average_daily_profile = df_all.groupby('hour')['value'].mean().reset_index()
# average_daily_profile.columns = ['hour_of_day', 'avg_load_kWh']

# # Print first few lines to check
# print("Daily load profile (first 5 days):")
# print(daily_profile.head())

# print("\nAverage daily load profile by hour:")
# print(average_daily_profile)

# # Total average energy inputted per day (in kWh)
# avg_daily_energy = average_daily_profile['avg_load_kWh'].sum()
# print(f"⚡ Average total energy inputted per day: {avg_daily_energy:.2f} kWh")

# # Optional: Plot daily load profile
# plt.figure(figsize=(12, 4))
# plt.plot(daily_profile['date'], daily_profile['daily_load_kWh'], label='Daily Load (kWh)')
# plt.xlabel('Date')
# plt.ylabel('Total Daily Load (kWh)')
# plt.title('Daily Load Profile for County 06033')
# plt.tight_layout()
# plt.show()

# # Optional: Plot average daily load profile by hour
# plt.figure(figsize=(8, 4))
# plt.plot(average_daily_profile['hour_of_day'], average_daily_profile['avg_load_kWh'], marker='o')
# plt.xlabel('Hour of Day')
# plt.ylabel('Average Load (kWh)')
# plt.title('Average Daily Load Profile by Hour for County 06033')
# plt.grid(True)
# plt.tight_layout()
# plt.show()

         date  hour  n_entries  hourly_total_value
0  2012-01-01     5          1                 0.0
1  2012-01-01     6          1                 0.0
2  2012-01-01     7          1                 0.0
3  2012-01-01     8          1                 0.0
4  2012-01-01     9          1                 0.0
5  2012-01-01    10          1                 0.0
6  2012-01-01    11          1                 0.0
7  2012-01-01    12          1                 0.0
8  2012-01-01    13          1                 0.0
9  2012-01-01    14          1                 0.0
Total unique (date, hour) combinations: 8784


In [99]:
# -----------------------
# County FIPS -> Name
# -----------------------
county_codes = {
    "06001": "Alameda County",
    "06003": "Alpine County", 
    "06005": "Amador County",
    "06007": "Butte County",
    "06009": "Calaveras County",
    "06011": "Colusa County",
    "06013": "Contra Costa County",
    "06015": "Del Norte County",
    "06017": "El Dorado County",
    "06019": "Fresno County",
    "06021": "Glenn County",
    "06023": "Humboldt County",
    "06025": "Imperial County",
    "06027": "Inyo County",
    "06029": "Kern County",
    "06031": "Kings County",
    "06033": "Lake County",
    "06035": "Lassen County",
    "06037": "Los Angeles County",
    "06039": "Madera County",
    "06041": "Marin County",
    "06043": "Mariposa County",
    "06045": "Mendocino County",
    "06047": "Merced County",
    "06049": "Modoc County",
    "06051": "Mono County",
    "06053": "Monterey County",
    "06055": "Napa County",
    "06057": "Nevada County",
    "06059": "Orange County",
    "06061": "Placer County",
    "06063": "Plumas County",
    "06065": "Riverside County",
    "06067": "Sacramento County",
    "06069": "San Benito County",
    "06071": "San Bernardino County",
    "06073": "San Diego County",
    "06075": "San Francisco County",
    "06077": "San Joaquin County",
    "06079": "San Luis Obispo County",
    "06081": "San Mateo County",
    "06083": "Santa Barbara County",
    "06085": "Santa Clara County",
    "06087": "Santa Cruz County",
    "06089": "Shasta County",
    "06091": "Sierra County",
    "06093": "Siskiyou County",
    "06095": "Solano County",
    "06097": "Sonoma County",
    "06099": "Stanislaus County",
    "06101": "Sutter County",
    "06103": "Tehama County",
    "06105": "Trinity County",
    "06107": "Tulare County",
    "06109": "Tuolumne County",
    "06111": "Ventura County",
    "06113": "Yolo County",
    "06115": "Yuba County"
}

In [ ]:
import pandas as pd
import os
from glob import glob

folder_path = './nrel_tempo_ca_2024'
file_paths = glob(os.path.join(folder_path, '*.parquet'))

target_county = '06001'
end_use = "electricity_ev_l1l2"
type = "Some_Drivers_Smaller+Middle_Income+Small_Town+SUV+BEV_300"

dfs = []
for path in file_paths:
    df = pd.read_parquet(path)
    df_filtered = df[ # (df['county'] == target_county) &
                     (df['end_use'] == end_use) &
                     (df['household_and_vehicle_type'] == type)
                    ]
    if not df_filtered.empty:
        dfs.append(df_filtered)

df_all = pd.concat(dfs, ignore_index=True)

# Ensure datetime and hourly alignment
df_all['time_est'] = pd.to_datetime(df_all['time_est'])
df_all['time_hour'] = df_all['time_est'].dt.floor('H')

# Restrict to California counties (FIPS '06***')
df_ca = df_all[df_all['county'].astype(str).str.startswith('06')].copy()

first_hour = df_ca['time_hour'].min()

n_entries_first_combo = (
    df_ca.loc[df_ca['time_hour'] == first_hour]
         .groupby('county')
         .size()
         .rename('n_entries')
         .reset_index()
         .sort_values('county')
         .reset_index(drop=True)
)

n_entries_first_combo['county_name'] = n_entries_first_combo['county'].replace(county_codes)

yearly_sum = (
    df_ca.groupby('county')['value']
         .sum()
         .rename('yearly_value_sum')
         .reset_index()
)

n_entries_first_combo = n_entries_first_combo.merge(
    yearly_sum,
    on='county',
    how='left'
)

print(f"First date+hour in CA data: {first_hour}")
print(n_entries_first_combo.head(20))

First date+hour in CA data: 2012-01-01 05:00:00
   county  n_entries             county_name  yearly_value_sum
0   06001          1          Alameda County         62.239066
1   06013          1     Contra Costa County        367.316194
2   06023          1         Humboldt County        214.253289
3   06045          1        Mendocino County        361.156417
4   06055          1             Napa County         73.001854
5   06059          1           Orange County        146.000549
6   06061          1           Placer County        175.925452
7   06067          1       Sacramento County        463.993962
8   06079          1  San Luis Obispo County       1755.165214
9   06081          1        San Mateo County        246.627348
10  06085          1      Santa Clara County        131.579773
11  06089          1           Shasta County        297.682813
12  06113          1             Yolo County        228.696362


In [ ]:
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt

# Path to all 20 files
folder_path = './nrel_tempo_ca_2024'
file_paths = glob(os.path.join(folder_path, '*.parquet'))

# Target county and vehicle type
target_county = '06001'
vehicle_filter = 'Midsize'

# Collect relevant rows from all files
dfs = []
for path in file_paths:
    df = pd.read_parquet(path, columns=['time_est', 'county', 'value', 'household_and_vehicle_type'])
    df_filtered = df[
        (df['county'] == target_county) &
        (df['household_and_vehicle_type'].str.contains(vehicle_filter))
    ]
    if not df_filtered.empty:
        dfs.append(df_filtered)

# Combine all data
df_all = pd.concat(dfs, ignore_index=True)

# YEARLY LOAD PROFILE (by day)
df_all['date'] = pd.to_datetime(df_all['time_est']).dt.date
yearly_profile = df_all.groupby('date')['value'].sum().reset_index()
yearly_profile.columns = ['date', 'daily_load_kWh']

# AVERAGE DAILY LOAD PROFILE (by hour)
df_all['hour'] = pd.to_datetime(df_all['time_est']).dt.hour
average_daily_profile = df_all.groupby('hour')['value'].mean().reset_index()
average_daily_profile.columns = ['hour_of_day', 'avg_load_kWh']

# Calculate total average daily energy input
avg_daily_energy = average_daily_profile['avg_load_kWh'].sum()
print(f"⚡ Average total energy inputted per day (Midsize vehicles, county {target_county}): {avg_daily_energy:.2f} kWh")

# --- Plotting ---

# Plot Yearly Load Profile
plt.figure(figsize=(12, 4))
plt.plot(yearly_profile['date'], yearly_profile['daily_load_kWh'], label='Daily Load (kWh)')
plt.title(f'Yearly Load Profile – County {target_county} – {vehicle_filter} Vehicles')
plt.xlabel('Date')
plt.ylabel('Daily Load (kWh)')
plt.tight_layout()
plt.grid(True)
plt.show()

# Plot Average Daily Load Profile
plt.figure(figsize=(8, 4))
plt.plot(average_daily_profile['hour_of_day'], average_daily_profile['avg_load_kWh'], marker='o')
plt.title(f'Average Daily Load – County {target_county} – {vehicle_filter} Vehicles')
plt.xlabel('Hour of Day')
plt.ylabel('Average Load (kWh)')
plt.xticks(range(0, 24))
plt.grid(True)
plt.tight_layout()
plt.show()

Test to look at the values on excel

In [ ]:
# import pandas as pd
# import os
# from glob import glob

# folder_path = './nrel_tempo_ca_2024'
# file_paths = glob(os.path.join(folder_path, '*.parquet'))

# target_county = '06001'  # Alameda
# target_vehicle_type = 'BEV'
# target_end_use = 'electricity_ev_l1l2'

# dfs = []
# for path in file_paths:
#     print(f"Reading {path} ...")
#     df = pd.read_parquet(path)

#     # Filter by county, vehicle type, and end use
#     df_filtered = df[
#         (df['county'] == target_county) &
#         (df['household_and_vehicle_type'].str.contains(target_vehicle_type)) &
#         (df['end_use'].str.contains(target_end_use))
#     ]

#     if not df_filtered.empty:
#         dfs.append(df_filtered)

# if not dfs:
#     print("No matching data found.")
# else:
#     df_all = pd.concat(dfs, ignore_index=True)
    
#     # Convert time_est to datetime, if not already
#     df_all['time_est'] = pd.to_datetime(df_all['time_est'])
    
#     # Earliest and latest timestamps for Alameda BEV electricity_ev_l1l2
#     earliest_time = df_all['time_est'].min()
#     latest_time = df_all['time_est'].max()
#     print(f"📅 Earliest timestamp: {earliest_time}")
#     print(f"📅 Latest timestamp: {latest_time}")

#     # Filter for first 7 days of the year (2024 assumed, but in excel we have 2012)
#     start_date = pd.Timestamp('2012-03-01')
#     end_date = pd.Timestamp('2012-03-07 23:59:59')
#     df_7days = df_all[(df_all['time_est'] >= start_date) & (df_all['time_est'] <= end_date)]

#     # Extract hour of day and day for grouping
#     df_7days['hour'] = df_7days['time_est'].dt.hour
#     df_7days['day'] = df_7days['time_est'].dt.date

#     # Group by day and hour to get sum or average load per hour per day
#     # (use sum if total load, mean if per vehicle average)
#     hourly_loads = df_7days.groupby(['day', 'hour'])['value'].mean().unstack(level=0)
    
#     # This gives a DataFrame with hours as rows (0-23), and columns as each day (Jan 1 to Jan 7)
#     print(hourly_loads)

#     # Optional: Plotting the 7 daily profiles
#     import matplotlib.pyplot as plt

#     plt.figure(figsize=(12, 6))
#     for day in hourly_loads.columns:
#         plt.plot(hourly_loads.index, hourly_loads[day], label=str(day))
#     plt.xlabel('Hour of Day')
#     plt.ylabel('Load (kWh)')
#     plt.title('Hourly Load Profiles for Alameda BEV - electricity_ev_l1l2 (First 7 Days of 2024)')
#     plt.legend(title='Date')
#     plt.grid(True)
#     plt.show()


Issue with time stamp, says 2012 for the data which should've been pulled for 2024. I will download the 2050 values and see if we have the same issue. 

In [ ]:
# import boto3
# from botocore import UNSIGNED
# from botocore.config import Config
# import os

# # Set up the S3 client with unsigned (public) access
# s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# bucket_name = 'nrel-pds-dsgrid'

# # ✅ 2050 data path in S3 bucket
# prefix = 'tempo/tempo-2022/v1.0.0/full_dataset/table.parquet/scenario=reference/tempo_project_model_years=2050/state=CA/'

# # Local folder to save files
# local_dir = './nrel_tempo_ca_2050'
# os.makedirs(local_dir, exist_ok=True)

# # List objects under the prefix
# response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# if 'Contents' not in response:
#     print("❌ No files found.")
# else:
#     for obj in response['Contents']:
#         key = obj['Key']
#         if key.endswith('.parquet'):
#             filename = os.path.basename(key)
#             local_path = os.path.join(local_dir, filename)
#             print(f"⬇️  Downloading {key} to {local_path}")
#             s3.download_file(bucket_name, key, local_path)

#     print("✅ All 2050 files downloaded.")


Check the max and min time stamsp for 2050 califonria reference scenario

In [ ]:
# import pandas as pd
# import os
# from glob import glob

# folder_path = './nrel_tempo_ca_2050'
# file_paths = glob(os.path.join(folder_path, '*.parquet'))

# target_county = '06001'  # Alameda

# all_timestamps = []

# for path in file_paths:
#     print(f"⏳ Reading {path} ...")
#     df = pd.read_parquet(path, columns=['county', 'time_est'])

#     df_filtered = df[df['county'] == target_county]

#     if not df_filtered.empty:
#         all_timestamps.extend(df_filtered['time_est'])

# # Convert to datetime and report min/max
# if all_timestamps:
#     timestamps = pd.to_datetime(all_timestamps)
#     print(f"\n📅 Earliest timestamp for Alameda (06001): {timestamps.min()}")
#     print(f"📅 Latest timestamp for Alameda (06001):   {timestamps.max()}")
# else:
#     print("⚠️ No data found for county 06001.")


Checking the difference in EV load for the same day in 2024 and 2050.

In [ ]:
# import pandas as pd
# import os
# import matplotlib.pyplot as plt
# from glob import glob

# def load_filtered_data(folder_path, target_county, target_vehicle_type, target_end_use):
#     file_paths = glob(os.path.join(folder_path, '*.parquet'))
#     dfs = []
#     for path in file_paths:
#         df = pd.read_parquet(path)
#         df_filtered = df[
#             (df['county'] == target_county) &
#             (df['household_and_vehicle_type'].str.contains(target_vehicle_type)) &
#             (df['end_use'].str.contains(target_end_use))
#         ]
#         if not df_filtered.empty:
#             dfs.append(df_filtered)
#     if dfs:
#         df_all = pd.concat(dfs, ignore_index=True)
#         df_all['time_est'] = pd.to_datetime(df_all['time_est'])
#         return df_all
#     else:
#         return None

# # Parameters
# target_county = '06001'  # Alameda
# target_vehicle_type = 'BEV'
# target_end_use = 'electricity_ev_l1l2'
# target_date = '2012-05-05'

# # Load both datasets
# df_2024 = load_filtered_data('./nrel_tempo_ca_2024', target_county, target_vehicle_type, target_end_use)
# print(df_2024["value"].sum())
# df_2050 = load_filtered_data('./nrel_tempo_ca_2050', target_county, target_vehicle_type, target_end_use)

# # Filter to May 5th only
# def extract_day(df, date_str):
#     return df[df['time_est'].dt.date == pd.to_datetime(date_str).date()]

# df_2024_day = extract_day(df_2024, target_date)
# df_2050_day = extract_day(df_2050, target_date)

# # Group by hour
# df_2024_day['hour'] = df_2024_day['time_est'].dt.hour
# df_2050_day['hour'] = df_2050_day['time_est'].dt.hour

# hourly_2024 = df_2024_day.groupby('hour')['value'].mean()
# hourly_2050 = df_2050_day.groupby('hour')['value'].mean()

# # Plotting
# plt.figure(figsize=(10, 6))
# plt.plot(hourly_2024.index, hourly_2024.values, label='2024', marker='o')
# plt.plot(hourly_2050.index, hourly_2050.values, label='2050', marker='s')
# plt.title('Hourly BEV Load on May 5 (Alameda, electricity_ev_l1l2)')
# plt.xlabel('Hour of Day')
# plt.ylabel('Average Load per Vehicle (kWh)')
# plt.legend()
# plt.grid(True)
# plt.xticks(range(0, 24))
# plt.tight_layout()
# plt.show()


Printing the daily load profiles for a week in january and for a week in june, with the total kWh.

In [ ]:
# import pandas as pd
# import os
# from glob import glob
# import matplotlib.pyplot as plt

# # --- Load Data ---
# folder_path = './nrel_tempo_ca_2024'
# file_paths = glob(os.path.join(folder_path, '*.parquet'))

# target_county = '06001'  # Alameda
# target_vehicle_type = 'BEV'
# target_end_use = 'electricity_ev_l1l2'

# dfs = []
# for path in file_paths:
#     df = pd.read_parquet(path)
#     df_filtered = df[
#         (df['county'] == target_county) &
#         (df['household_and_vehicle_type'].str.contains(target_vehicle_type)) &
#         (df['end_use'] == target_end_use)
#     ]
#     if not df_filtered.empty:
#         dfs.append(df_filtered)

# if not dfs:
#     print("No matching data found.")
#     exit()

# df_all = pd.concat(dfs, ignore_index=True)
# df_all['time_est'] = pd.to_datetime(df_all['time_est'])

# # --- Define target days ---
# week1_dates = pd.date_range('2012-01-02', '2012-01-08').tolist()
# week2_dates = pd.date_range('2012-06-04', '2012-06-10').tolist()
# target_dates = week1_dates + week2_dates

# daily_totals = []

# # --- Create Plot Grid ---
# fig, axes = plt.subplots(nrows=2, ncols=7, figsize=(20, 8), sharey=True)
# axes = axes.flatten()

# for i, target_day in enumerate(target_dates):
#     # Filter 24-hour period
#     day_data = df_all[df_all['time_est'].dt.date == target_day.date()]
#     hourly_avg = day_data.groupby(day_data['time_est'].dt.hour)['value'].mean()
#     daily_kwh = hourly_avg.sum()
#     daily_totals.append((target_day, daily_kwh))

#     # Plot
#     axes[i].plot(hourly_avg.index, hourly_avg.values, marker='o')
#     axes[i].set_title(target_day.strftime('%a %b %d'))
#     axes[i].set_xlabel('Hour')
#     axes[i].set_xlim(0, 23)
#     if i % 7 == 0:
#         axes[i].set_ylabel('Avg Load (kWh)')
#     axes[i].grid(True)

# plt.suptitle('Hourly Load Profiles (BEV, Alameda, electricity_ev_l1l2, 2024)', fontsize=16)
# plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.show()

# # --- Print Daily and Weekly Totals ---
# print("\n🔋 Daily Total kWh Consumed (Average per Vehicle):")
# week1_total = 0
# week2_total = 0

# for i, (day, total_kwh) in enumerate(daily_totals):
#     print(f"{day.strftime('%Y-%m-%d')} : {total_kwh:.2f} kWh")
#     if i < 7:
#         week1_total += total_kwh
#     else:
#         week2_total += total_kwh

# print(f"\n📅 Week 1 Total (Jan 2–8): {week1_total:.2f} kWh")
# print(f"📅 Week 2 Total (June 4–10): {week2_total:.2f} kWh")
